In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:(Plot the target distribution (delivery_time))
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:(Drop the 'Order_ID' column from the data)
df=df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here: (Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df[cols].copy()

In [ ]:
# Drop rows where target (delivery time) - can't predict without it
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time'])
print(f"After dropping missing Delivery Time: {df_clean.shape}")

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

In [ ]:
# Fill years of experience with mode - discrete feature, mode is most representative
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here: (Check and remove duplicates if any exist)
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) #to have it in the original data
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here: (Encode categorical variables if needed (Bonus if used One Hot Encoding))
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 5: Write your code here: (Apply feature scaling for all features (Use StandardScaler))
from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop("Delivery_Time")

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here: (Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed))

In [ ]:
# Task 1: Write your code here:(Split the dataset into features (X) and target (y))
X = df_clean.drop(columns=['Delivery_Time']).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here: ( 1-Use the correct split: KFold OR StratifiedKFold 2-Train a RandomForest model
#3- Evaluate using MAE (Mean Absolute Error) ONLY 4-Print the averaged score across all folds)
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

n_splits = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_mae = []

model = RandomForestRegressor(n_estimators=200)


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  lr_mae.append(mae)

print(f"  MAE:  {np.mean(lr_mae, axis=0):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature':  X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here: (Plot predicted delivery time histogram)
plt.figure(figsize=(6, 4))
plt.hist(y_pred, bins=30, edgecolor='black', color='pink')
plt.title("predicted delivery time histogram")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: